In [ ]:
# Cell 1 — Install dependencies
pip install pandukabhaya langchain langchain-community langchain-huggingface faiss-cpu pymupdf sentence-transformers

In [ ]:
import os
import re
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from pandukabhaya import Converter # Sinhala Legacy to Unicode Converter

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH  = "data_s/"          # folder containing your Sinhala PDFs
INDEX_PATH = "faiss_index_s"    # output folder — zip this and bring offline

# Must use a multilingual model for Sinhala text!
EMBED_MODEL = "sentence-transformers/LaBSE"

# CHUNKING CONFIGURATION
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 200

# Initialize the Sinhala Converter
fm_converter = Converter("fm_abhaya")

# --- HELPER: Heal the PDF Text ---
def clean_pdf_text(text: str) -> str:
    """Removes arbitrary PDF line breaks while preserving actual paragraphs."""
    if not text:
        return ""
    # Replace single newlines with spaces (heals broken sentences)
    # but strictly preserves double newlines (actual paragraphs)
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    # Clean up any weird multiple spaces left behind
    text = re.sub(r' +', ' ', text)
    return text.strip()

# ── Step 1: Load, Convert & Heal PDFs ─────────────────────────────────────────
def load_documents(data_path: str) -> list[Document]:
    documents = []
    pdf_files = [f for f in os.listdir(data_path) if f.endswith(".pdf")]

    for fname in sorted(pdf_files):
        fpath = os.path.join(data_path, fname)
        print(f"  Loading, Converting & Cleaning: {fname}", end=" ... ")
        loader = PyMuPDFLoader(fpath)
        pages = loader.load()

        for page in pages:
            page.metadata["source"] = fname

            # 1. Convert Legacy Sinhala font to standard Unicode
            converted_text = fm_converter.convert(page.page_content)

            # 2. Heal the text (remove random PDF line breaks)
            page.page_content = clean_pdf_text(converted_text)

        documents.extend(pages)
        print(f"{len(pages)} pages processed")

    return documents

# ── Step 2: Sanity check (Sinhala) ────────────────────────────────────────────
def sanity_check(documents: list[Document]) -> bool:
    sample = documents[0].page_content[:400] if documents else ""
    print("\n--- SANITY CHECK (first 400 chars of page 1) ---")
    print(sample)
    print("--- END ---\n")

    # Check if standard Sinhala Unicode blocks exist in the text
    has_sinhala = any('\u0d80' <= ch <= '\u0dff' for ch in sample)
    if not has_sinhala:
        print("⚠️ WARNING: Still no Sinhala characters detected!")
        return False

    print("✅ Sinhala Unicode successfully converted, healed, and detected!")
    return True

# ── Step 3: Chunking ──────────────────────────────────────────────────────────
def calculate_chunk_ids(chunks: list[Document]) -> list[Document]:
    last_page_id = None
    current_chunk_index = 0
    for chunk in chunks:
        source = chunk.metadata.get("source", "unknown")
        page   = chunk.metadata.get("page", 0)
        current_page_id = f"{source}:{page}"
        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0
        chunk.metadata["id"]   = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id
    return chunks

def split_documents(documents: list[Document]) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        # Tweaked separators for Sinhala punctuation
        separators=["\n\n", ".\n", ".", " ", ""],
        length_function=len,
    )
    raw_chunks = splitter.split_documents(documents)

    # --- JUNK FILTER ---
    clean_chunks = []
    for chunk in raw_chunks:
        text = chunk.page_content.strip()
        # Drop chunks that are extremely short or lack spaces
        if len(text) > 30 and " " in text:
            clean_chunks.append(chunk)

    print(f"🧹 Filtered out {len(raw_chunks) - len(clean_chunks)} junk chunks.")
    return calculate_chunk_ids(clean_chunks)

# ── Step 4: Embed and save to FAISS ───────────────────────────────────────────
def add_to_faiss(chunks: list[Document], embeddings: HuggingFaceEmbeddings) -> None:
    print(f"✨ Creating new FAISS index...")
    ids = [c.metadata["id"] for c in chunks]
    db = FAISS.from_documents(chunks, embeddings, ids=ids)
    db.save_local(INDEX_PATH)
    print(f"✅ Index saved to '{INDEX_PATH}/'")

# ── Main ──────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1: Loading, Converting & Cleaning PDFs")
print("=" * 60)
documents = load_documents(DATA_PATH)
print(f"\nTotal pages loaded: {len(documents)}")

ok = sanity_check(documents)
if not ok:
    raise RuntimeError("Conversion failed. Check if PDFs contain readable text.")

print("\n" + "=" * 60)
print("STEP 2: Chunking")
print("=" * 60)
chunks = split_documents(documents)
print(f"Total chunks: {len(chunks)}")

print("\n" + "=" * 60)
print("STEP 3: Loading embedding model")
print("=" * 60)
print(f"\nDownloading {EMBED_MODEL} Model...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={'normalize_embeddings': True} # Keeps the accuracy boost for LaBSE!
)

print("\n" + "=" * 60)
print("STEP 4: Building FAISS index")
print("=" * 60)
add_to_faiss(chunks, embeddings)
print("\n🎉 DONE! Zip the 'faiss_index' folder and download it.")

In [ ]:
# Run this cell first to install dependencies in Colab
pip install pymupdf langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers

In [ ]:
import os
import re
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH  = "data/"          # folder containing your English PDFs
INDEX_PATH = "faiss_index"    # output folder — zip this and bring offline

# UPGRADED CHUNKING CONFIGURATION
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 200

# --- NEW HELPER: Heal the PDF Text ---
def clean_pdf_text(text: str) -> str:
    """Removes arbitrary PDF line breaks while preserving actual paragraphs."""
    if not text:
        return ""
    # Replace single newlines with spaces (heals broken sentences)
    # but strictly preserves double newlines (actual paragraphs)
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    # Clean up any weird multiple spaces left behind
    text = re.sub(r' +', ' ', text)
    return text.strip()

# ── Step 1: Load PDFs (Upgraded) ──────────────────────────────────────────────
def load_documents(data_path: str) -> list[Document]:
    documents = []
    pdf_files = [f for f in os.listdir(data_path) if f.endswith(".pdf")]

    for fname in sorted(pdf_files):
        fpath = os.path.join(data_path, fname)
        print(f"  Loading: {fname}", end=" ... ")
        loader = PyMuPDFLoader(fpath)
        pages = loader.load()

        for page in pages:
            page.metadata["source"] = fname
            # HEAL THE TEXT before adding it to the document list
            page.page_content = clean_pdf_text(page.page_content)

        documents.extend(pages)
        print(f"{len(pages)} pages processed")

    return documents

# ── Step 2: Sanity check (English) ────────────────────────────────────────────
def sanity_check(documents: list[Document]) -> bool:
    sample = documents[0].page_content[:400] if documents else ""
    print("\n--- SANITY CHECK (first 400 chars of page 1) ---")
    print(sample)
    print("--- END ---\n")

    if not sample.strip():
        print("⚠️ WARNING: No text detected! The PDF might be scanned images.")
        return False

    print("✅ Text successfully loaded and healed!")
    return True

# ── Step 3: Chunking (Upgraded) ───────────────────────────────────────────────
def calculate_chunk_ids(chunks: list[Document]) -> list[Document]:
    last_page_id = None
    current_chunk_index = 0
    for chunk in chunks:
        source = chunk.metadata.get("source", "unknown")
        page   = chunk.metadata.get("page", 0)
        current_page_id = f"{source}:{page}"
        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0
        chunk.metadata["id"]   = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id
    return chunks

def split_documents(documents: list[Document]) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        # UPGRADED SEPARATORS: Prioritize paragraphs, then strict sentence ends
        separators=["\n\n", ". ", "? ", "! ", " "],
        length_function=len,
    )
    raw_chunks = splitter.split_documents(documents)

    # --- JUNK FILTER ---
    clean_chunks = []
    for chunk in raw_chunks:
        text = chunk.page_content.strip()
        # Drop chunks that are extremely short or lack spaces
        if len(text) > 50 and " " in text:
            clean_chunks.append(chunk)

    print(f"🧹 Filtered out {len(raw_chunks) - len(clean_chunks)} junk chunks.")
    return calculate_chunk_ids(clean_chunks)

# ── Step 4: Embed and save to FAISS ───────────────────────────────────────────
def add_to_faiss(chunks: list[Document], embeddings: HuggingFaceEmbeddings) -> None:
    print(f"✨ Creating new FAISS index...")
    ids = [c.metadata["id"] for c in chunks]
    db = FAISS.from_documents(chunks, embeddings, ids=ids)
    db.save_local(INDEX_PATH)
    print(f"✅ Index saved to '{INDEX_PATH}/'")

# ── Main ──────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1: Loading & Cleaning PDFs")
print("=" * 60)
documents = load_documents(DATA_PATH)
print(f"\nTotal pages loaded: {len(documents)}")

ok = sanity_check(documents)
if not ok:
    raise RuntimeError("Loading failed. Check if PDFs are readable text.")

print("\n" + "=" * 60)
print("STEP 2: Chunking")
print("=" * 60)
chunks = split_documents(documents)
print(f"Total chunks: {len(chunks)}")

print("\n" + "=" * 60)
print("STEP 3: Loading embedding model")
print("=" * 60)
print("\nDownloading Model")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={'normalize_embeddings': True} # <-- This is the secret to better scores!
)

print("\n" + "=" * 60)
print("STEP 4: Building FAISS index")
print("=" * 60)
add_to_faiss(chunks, embeddings)
print("\n🎉 DONE! Zip the 'faiss_index' folder and download it.")